# CDLD training for microbial coculture prediction

Train the discovery models and prediction models, select models using validation data, and evaluate performance across training-data fractions. The cells use the model code and data splits in full/ and fractions/.
Run 00_Prepare_Data.ipynb first. Outputs are generated when this source notebook is executed.

In [ ]:
from pathlib import Path
import os,sys,json,time,subprocess,concurrent.futures,platform
R=Path.cwd();assert (R/'full/primary/model.py').exists()
assert platform.system()=='Darwin', 'This run is designated macOS reproduction'
SEEDS=[17,43,101,137,211]
def run(args):
 env=os.environ.copy();env.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',VECLIB_MAXIMUM_THREADS='2')
 tag='_'.join(str(x).replace('/','_') for x in args)[-170:];logs=R/'logs';logs.mkdir(exist_ok=True)
 with (logs/(tag+'.log')).open('a') as f:
  subprocess.run([sys.executable,*map(str,args)],cwd=R,env=env,stdout=f,stderr=subprocess.STDOUT,check=True)
 return tag
def batch(jobs,label):
 print(label, 'jobs:',len(jobs),flush=True)
 with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
  futures=[pool.submit(run,j) for j in jobs]
  for k,f in enumerate(concurrent.futures.as_completed(futures),1):
   tag=f.result();print(label,k,'/',len(jobs),tag,flush=True)
   (R/'status.json').write_text(json.dumps(dict(phase=label,completed=k,total=len(jobs),updated=time.time())))


In [ ]:
batch([[R/'full'/sp/'run_job.py','source','gm',s,'source'] for sp in ['primary','robust'] for s in SEEDS], '24h_discovery')

In [ ]:
batch([[R/'full'/sp/'run_job.py','reader','gm',s,t] for sp in ['primary','robust'] for t in ['hour48','trio'] for s in SEEDS]+[[R/'full/primary/run_job.py','aug','gm',s,'source'] for s in SEEDS], 'full_predictors')

In [ ]:
batch([[R/'full/baselines.py',R/'full/primary']], 'primary_baselines')
batch([[R/'full/baselines.py',R/'full/robust']], 'robust_baselines')

In [ ]:
items=json.loads((R/'fractions/subsets.json').read_text())
# 100% LF is fitted again to verify the adapter's validation predictions.
batch([[R/'fractions/train.py','q100_r00','LF',s,'reader'] for s in SEEDS], '100pct_gate')
import numpy as np
for s in SEEDS:
 a=np.load(R/f'full/primary/runs/reader_hour48_gm_{s}/validation_predictions.npz')
 b=np.load(R/f'fractions/subsets/q100_r00/LF/runs/reader_hour48_initial_{s}/validation_predictions.npz')
 np.testing.assert_array_equal(a['row_id'],b['row_id']);np.testing.assert_allclose(a['prediction'],b['prediction'],rtol=0,atol=1e-7)
print('100% validation gate passed')

In [ ]:
batch([[R/'fractions/train.py',x['name'],c,s,'source'] for x in items for c in ['LU','RU'] for s in SEEDS], '48h_discovery')

In [ ]:
batch([[R/'fractions/train.py',x['name'],c,s,'reader'] for x in items for c in ['LF','LU','RU'] for s in SEEDS if not(x['pct']==100 and c=='LF')], '48h_predictors')

In [ ]:
items=json.loads((R/'fractions/subsets.json').read_text())
batch([[R/'fractions/baselines.py',R/'fractions/subsets'/x['name']] for x in items if x['pct']!=100], 'fraction_baselines')

In [ ]:
run([R/'fractions/evaluate.py'])
import pandas as pd
metrics=pd.read_json(R/'fractions/evaluation/metrics.json')
display(metrics.groupby(['pct','model']).mse.mean().unstack())
display(pd.read_json(R/'fractions/evaluation/contrasts.json'))
(R/'status.json').write_text(json.dumps(dict(phase='complete',updated=time.time())))